<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/AgentRouter_Lite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AgentRouter Lite: Route Queries to the Right Tool (Colab)

Pipeline:
User Query
→ Intent Detection
→ Tool Selection
→ Tool Execution
→ Final Response + Routing Trace

Tools:
- SQL tool
- RAG tool
- Calculator tool
- Generic text tool

Goals:
- Show agentic orchestration
- Keep routing explainable
- Run fully in Colab with no API key

In [3]:
#!pip -q install duckdb pandas

In [2]:
import re
import math
import duckdb
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Optional, List

In [4]:
data = [
    {"app_name": "Atlas",   "team": "Platform", "director": "Patel",  "cost_usd_per_month": 12000, "incidents_last_30d": 2, "monthly_active_users": 40000},
    {"app_name": "Nimbus",  "team": "Payments", "director": "Chen",   "cost_usd_per_month": 22000, "incidents_last_30d": 5, "monthly_active_users": 70000},
    {"app_name": "Orion",   "team": "Core",     "director": "Singh",  "cost_usd_per_month": 18000, "incidents_last_30d": 1, "monthly_active_users": 55000},
    {"app_name": "Pulse",   "team": "Mobile",   "director": "Garcia", "cost_usd_per_month": 14000, "incidents_last_30d": 4, "monthly_active_users": 62000},
    {"app_name": "Beacon",  "team": "Data",     "director": "Nguyen", "cost_usd_per_month": 26000, "incidents_last_30d": 3, "monthly_active_users": 81000},
]

df_apps = pd.DataFrame(data)

con = duckdb.connect(database=":memory:")
con.execute("CREATE TABLE apps AS SELECT * FROM df_apps")
con.execute("SELECT * FROM apps").fetchdf()

,app_name,team,director,cost_usd_per_month,incidents_last_30d,monthly_active_users
0,Atlas,Platform,Patel,12000,2,40000
1,Nimbus,Payments,Chen,22000,5,70000
2,Orion,Core,Singh,18000,1,55000
3,Pulse,Mobile,Garcia,14000,4,62000
4,Beacon,Data,Nguyen,26000,3,81000


In [5]:
DOCS = [
    {
        "doc_id": "doc_1",
        "title": "RAG Design Notes",
        "text": """
        Retrieval-Augmented Generation (RAG) combines document retrieval with answer generation.
        The retriever finds relevant chunks from a knowledge base, and the generator uses those chunks
        to answer questions more faithfully. Citations improve trust and make answers easier to verify.
        """
    },
    {
        "doc_id": "doc_2",
        "title": "Text-to-SQL Guardrails",
        "text": """
        Text-to-SQL systems should use read-only queries, table allowlists, column allowlists, query limits,
        and timeouts. They should also log generated SQL and reject dangerous statements like DROP, DELETE, or UPDATE.
        """
    },
    {
        "doc_id": "doc_3",
        "title": "LLM Evaluation Basics",
        "text": """
        Evaluation frameworks for LLM systems often score correctness, completeness, clarity, and faithfulness.
        Pairwise comparisons are useful when direct scoring is noisy.
        """
    }
]

In [6]:
def sql_tool(query: str) -> Dict[str, Any]:
    q = query.lower()

    if "highest cost" in q or ("cost" in q and "highest" in q):
        sql = "SELECT app_name, cost_usd_per_month FROM apps ORDER BY cost_usd_per_month DESC LIMIT 5;"
    elif "most incidents" in q or ("incidents" in q and "top" in q):
        sql = "SELECT app_name, incidents_last_30d FROM apps ORDER BY incidents_last_30d DESC LIMIT 5;"
    elif "by team" in q and ("count" in q or "how many" in q):
        sql = "SELECT team, COUNT(*) AS n FROM apps GROUP BY team ORDER BY n DESC;"
    elif "monthly active users" in q or "users" in q:
        sql = "SELECT app_name, monthly_active_users FROM apps ORDER BY monthly_active_users DESC LIMIT 5;"
    else:
        sql = "SELECT * FROM apps LIMIT 5;"

    result = con.execute(sql).fetchdf()
    return {
        "tool": "sql_tool",
        "sql": sql,
        "result": result
    }

In [7]:
def tokenize(text: str) -> set:
    toks = re.findall(r"[a-z0-9_]+", text.lower())
    stop = {"the","a","an","and","or","to","of","in","on","for","with","is","are","was","were","be","it","that","this"}
    return {t for t in toks if t not in stop and len(t) > 2}

def rag_tool(query: str, top_k: int = 2) -> Dict[str, Any]:
    q_tokens = tokenize(query)
    scored = []

    for doc in DOCS:
        d_tokens = tokenize(doc["text"] + " " + doc["title"])
        overlap = len(q_tokens & d_tokens)
        scored.append((overlap, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_docs = [d for s, d in scored[:top_k] if s > 0]

    if not top_docs:
        answer = "I couldn't find relevant information in the document store."
        citations = []
    else:
        snippets = []
        citations = []
        for d in top_docs:
            snippet = " ".join(d["text"].split())[:220]
            snippets.append(f"- {snippet}")
            citations.append(d["title"])
        answer = "Here’s what I found:\n" + "\n".join(snippets)

    return {
        "tool": "rag_tool",
        "result": answer,
        "citations": citations
    }

In [8]:
ALLOWED_CHARS = set("0123456789+-*/(). %")

def calculator_tool(query: str) -> Dict[str, Any]:
    expr_match = re.findall(r"[\d\+\-\*\/\(\)\.\% ]+", query)
    expr = max(expr_match, key=len).strip() if expr_match else ""

    if not expr or any(ch not in ALLOWED_CHARS for ch in expr):
        return {
            "tool": "calculator_tool",
            "result": "No valid arithmetic expression found."
        }

    try:
        value = eval(expr, {"__builtins__": {}}, {})
        return {
            "tool": "calculator_tool",
            "expression": expr,
            "result": value
        }
    except Exception as e:
        return {
            "tool": "calculator_tool",
            "expression": expr,
            "result": f"Calculation failed: {e}"
        }

In [9]:
def generic_text_tool(query: str) -> Dict[str, Any]:
    cleaned = re.sub(r"\s+", " ", query).strip()
    if len(cleaned) < 120:
        result = f"Generic response: {cleaned}"
    else:
        result = f"Summary: {cleaned[:120]}..."
    return {
        "tool": "generic_text_tool",
        "result": result
    }

In [ ]:
@dataclass
class RouteDecision:
    intent: str
    tool_name: str
    confidence: float
    reason: str

def detect_intent(query: str) -> RouteDecision:
    q = query.lower()

    # SQL / data questions
    sql_keywords = [
        "sql", "table", "database", "query", "cost", "incidents",
        "monthly active users", "by team", "apps", "director"
    ]
    if any(k in q for k in sql_keywords) and any(w in q for w in ["show", "count", "how many", "top", "highest", "list"]):
        return RouteDecision(
            intent="data_query",
            tool_name="sql_tool",
            confidence=0.90,
            reason="Detected structured data request with analytics keywords."
        )

    # RAG / document QA
    rag_keywords = [
        "document", "pdf", "notes", "rag", "citation", "design notes",
        "what does the document say", "according to the docs"
    ]
    if any(k in q for k in rag_keywords):
        return RouteDecision(
            intent="document_qa",
            tool_name="rag_tool",
            confidence=0.88,
            reason="Detected document-oriented or knowledge-base question."
        )

    # calculator
    if re.search(r"\d+\s*[\+\-\*\/]\s*\d+", q) or "calculate" in q:
        return RouteDecision(
            intent="calculation",
            tool_name="calculator_tool",
            confidence=0.95,
            reason="Detected arithmetic expression or calculation intent."
        )

    # fallback
    return RouteDecision(
        intent="generic_text",
        tool_name="generic_text_tool",
        confidence=0.60,
        reason="No strong tool signal detected; using generic fallback."
    )

In [10]:
TOOLS = {
    "sql_tool": sql_tool,
    "rag_tool": rag_tool,
    "calculator_tool": calculator_tool,
    "generic_text_tool": generic_text_tool,
}

def run_agent(query: str) -> Dict[str, Any]:
    decision = detect_intent(query)
    tool_fn = TOOLS[decision.tool_name]
    output = tool_fn(query)

    return {
        "query": query,
        "route": {
            "intent": decision.intent,
            "tool_name": decision.tool_name,
            "confidence": decision.confidence,
            "reason": decision.reason,
        },
        "tool_output": output
    }